In [1]:
import os, sys

nb_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(nb_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Using project root:', project_root)
print('First sys.path entry:', sys.path[0])

Using project root: /Users/rithvik/Documents/hnrs/Decoder
First sys.path entry: /Users/rithvik/Documents/hnrs/Decoder


In [9]:
import numpy as np
from utils.LDPC_encode import QCLDPCEncoder

base_pc_matrix = '../pc_matrices/NR_1_0_32.txt'
P = np.loadtxt(base_pc_matrix, dtype=int)
blocksize = 32

encoder = QCLDPCEncoder(base_matrix= P, Z= blocksize)

Initializing Encoder: Full Matrix Size 1472x2176, Message Bits: 704
  > Inverting Parity Matrix (this may take a moment for large Z)...
  > Computing Generator Matrix...
Encoder Ready.


In [10]:
k = (P.shape[1] - P.shape[0]) * blocksize
n_frames = 1000

message = np.random.randint(0, 2, (n_frames, k))

codeword = encoder.encode(message)
tx_codeword = 1 - 2 * codeword

In [11]:
from ldpc.bp_decoder import BpDecoder

decoder = BpDecoder(encoder.H)
max_iter = 30

In [12]:
m = P.shape[0] * blocksize
check_nodes = np.arange(m)

clusters = check_nodes.reshape(P.shape[0], -1)

In [13]:
from utils.awgn_channel import AWGNChannel
from utils.find_ber import findBER
from utils.res_cluster_picker import pick_max_avg_residual_cluster, pick_max_max_residual_cluster

snrs = [4, 5, 6, 7, 8]
bers = []


for snr in snrs:
    rx_llrs = AWGNChannel(tx_codeword, snr_db=snr)
    decoded_codewords = []
    # schedule = []

    for i in range(n_frames):
        llr = rx_llrs[i, :]
        # times_cluster = [0, 0, 0, 0, 0, 0]

        decoder.reset()
        decoder.initialise_log_domain_bp(llr)
        
        for iter in range(max_iter):
            residuals = decoder.get_residuals()
            cluster_idx, scheduled_cluster = pick_max_avg_residual_cluster(residuals, clusters)
            # schedule.append(cluster_idx)

            # print(f"Scheduled cluster at iteration {iter}: {cluster_idx}")
            # times_cluster[cluster_idx] += 1

            llr = decoder.decode_cluster(scheduled_cluster)

        
        decoded_codeword = (llr < 0).astype(int)
        decoded_codewords.append(decoded_codeword)
        # print("Times each cluster was scheduled:", times_cluster)
    
    decoded_codewords = np.array(decoded_codewords)
    decoded_message = decoded_codewords[:, :k]
    ber = findBER(message, decoded_message)
    bers.append(ber)
    print(f"BER at SNR {snr} dB: {ber}")

    # print("scheduled clusters:", schedule)
    


BER at SNR 4 dB: 0.0034289772727272728
BER at SNR 5 dB: 0.0014801136363636364
BER at SNR 6 dB: 0.0006235795454545454
BER at SNR 7 dB: 0.00028267045454545456
BER at SNR 8 dB: 0.000125
